# Milestone 3 - LLM comparisons 

## Imports and Setup

In [12]:
import sys
import os
sys.path.append('../src')

from session_helper import init_session, load_model_and_index
from rag_pipeline import retrieve_hybrid, build_context, strip_thinking
from langchain_groq import ChatGroq
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from dotenv import load_dotenv
load_dotenv(dotenv_path='../.env')

con = init_session()                                 # initialize connection and semantic resources
model, index, metadata = load_model_and_index()

import warnings
import nltk
warnings.filterwarnings("ignore", category=FutureWarning)   # ignore [FutureWarning] warnings
nltk.download('punkt', quiet=True)                         # quiet [punkt] warnings

print("Session initialized.")

Session initialized.


## Define Models for Comparison => Ascending Number of Parameters

In [14]:
models = {
    "llama-3.1-8b":     ChatGroq(model="llama-3.1-8b-instant",    api_key=os.getenv("GROQ_API_KEY")),
    "gpt-oss-20b":      ChatGroq(model="openai/gpt-oss-20b",      api_key=os.getenv("GROQ_API_KEY")),
    "qwen3-32b":        ChatGroq(model="qwen/qwen3-32b",          api_key=os.getenv("GROQ_API_KEY")),
    "llama-3.3-70b":    ChatGroq(model="llama-3.3-70b-versatile", api_key=os.getenv("GROQ_API_KEY")),
}

print("Models initialized:", list(models.keys()))

Models initialized: ['llama-3.1-8b', 'gpt-oss-20b', 'qwen3-32b', 'llama-3.3-70b']


## Define Test Queries of Varying Difficulty Levels

In [15]:
test_queries = [
    "garden hose 50ft",                                      # easy => keyword
    "something to keep slugs off my plants",                  # medium => semantic
    "best fertilizer for tomatoes under $20",                  # medium => complex
    "durable outdoor furniture that survives harsh winters",    # complex
    "eco friendly pest control for vegetable garden",            # most complex
]

## Prompt Template for All Models

In [16]:
SYSTEM_PROMPT = """You are a helpful Amazon shopping assistant specializing 
in patio, lawn and garden products. Answer the question using ONLY the 
provided product context. Be concise and cite product names when possible. 
If the context does not contain enough information, say so."""

prompt_template = ChatPromptTemplate.from_messages([
    ("system", SYSTEM_PROMPT),
    ("human", "Context:\n{context}\n\nQuestion: {question}")
])

## Run All Queries Through All Models

In [17]:
results = {}

for query in test_queries:
    print(f"\nQuery: '{query}'")
    
    docs    = retrieve_hybrid(query, k=5)     # retrieve context once => reuse for all models
    context = build_context(docs)

    results[query] = {"context": context, "responses": {}}
    
    for model_name, llm in models.items():
        chain    = prompt_template | llm | StrOutputParser()
        response = strip_thinking(chain.invoke({
            "context":  context,
            "question": query
        }))
        results[query]["responses"][model_name] = response
        print(f"[{model_name}] done")


Query: 'garden hose 50ft'


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

[llama-3.1-8b] done
[gpt-oss-20b] done
[qwen3-32b] done
[llama-3.3-70b] done

Query: 'something to keep slugs off my plants'


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

[llama-3.1-8b] done
[gpt-oss-20b] done
[qwen3-32b] done
[llama-3.3-70b] done

Query: 'best fertilizer for tomatoes under $20'


...

[llama-3.1-8b] done
[gpt-oss-20b] done
[qwen3-32b] done
[llama-3.3-70b] done

Query: 'durable outdoor furniture that survives harsh winters'


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

[llama-3.1-8b] done
[gpt-oss-20b] done
[qwen3-32b] done
[llama-3.3-70b] done

Query: 'eco friendly pest control for vegetable garden'


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

[llama-3.1-8b] done
[gpt-oss-20b] done
[qwen3-32b] done
[llama-3.3-70b] done


## Print Results 

In [18]:
for query, data in results.items():
    print(f"\n{'='*80}")
    print(f"QUERY: {query}")
    print(f"{'='*80}")
    for model_name, response in data["responses"].items():
        print(f"\n--- {model_name} ---")
        print(response)


QUERY: garden hose 50ft

--- llama-3.1-8b ---
Based on the provided products, there are four options for a 50ft garden hose. 

1. Garden Hose Expandable 50ft, Water Hose Nozzle Sprayer 8 Way Spray Pattern Car Wash Foam Gun with Universal Faucet Connector for Watering Plants, Lawn, Patio, Cleaning, Showering Pet (B09XMZRZDY) - 2.3/5 rating
2. Komnn- Expandable Flexible Garden Hose Expanding Compact Lightweight Garden Hose with PREMIUM Brass Connectors, 8 Pattern Spray Nozzle and Hanger - 50 Feet (B073NTTQ75) - 2.7/5 rating
3. Garden Hoses 50ft Expandable Water Hose Patio Lawn Garden Accessories Car Wash Pet Cleaning 10 Function Spray Nozzles (B087Q4QGNW) - 4.3/5 rating
4. 2018 Upgraded Expandable Garden Hose,Best 50 Ft Flexible Water Hose with 9 High Pressure Spray Nozzle,Solid Brass Connector Fittings no Rust&Leak, Double Latex Core&Extra Strength Fabric(50FT) (black) (B07F9WCD3F) - 3.4/5 rating

If you would like to purchase a 50ft garden hose, I recommend considering the Garden Hose

## Save Comparison Results to a Markdown File

In [19]:
output_path = "../results/llm_comparison.md"

with open(output_path, 'w', encoding='utf-8') as f:
    f.write("# LLM Comparison Results\n\n")
    f.write("**Models compared:** qwen3-32b, llama-3.1-8b, llama-3.3-70b, gpt-oss-20b\n\n")
    f.write("**Retriever:** Hybrid (BM25 + Semantic)\n\n")
    f.write("**Prompt:** Identical across all models\n\n")
    f.write("---\n\n")

    for query, data in results.items():
        f.write(f"## Query: *{query}*\n\n")
        
        for model_name, response in data["responses"].items():
            f.write(f"### {model_name}\n\n")
            f.write(f"{response}\n\n")
        
        f.write("---\n\n")

print(f"Saved to {output_path}")

Saved to ../results/llm_comparison.md
